In [1]:
#!/usr/bin/env python3
"""
Phase 6: Interactive Reasoning Tracker - Jupyter Notebook
Automatically detects and verifies virtual environment status
"""

import sys
import os

# Pre-flight check: Show kernel info
print("\n" + "🚀 " + "=" * 58)
print("JUPYTER KERNEL STARTUP DIAGNOSTICS")
\
print("=" * 60)
print(f"Kernel Python: {sys.executable}")
print(f"Version: {sys.version}")
print(f"Platform: {sys.platform}")
print()

# Check if this is a venv kernel
is_venv = hasattr(sys, 'real_prefix') or (hasattr(sys, 'base_prefix') and sys.base_prefix != sys.prefix)
print(f"Running in Virtual Environment: {'✅ YES' if is_venv else '⚠️  NO'}")

if not is_venv:
    print()
    print("⚠️  HEADS UP: This kernel is not using a virtual environment!")
    print()
    print("RECOMMENDED: Restart and use the launcher script:")
    print("  $ cd phase6 && ./run_jupyter.sh")
    print()
    print("MANUAL FIX:")
    print("  1. Close this notebook")
    print("  2. In terminal, run:")
    print("     source venv/bin/activate")
    print("     jupyter notebook")
    print()
    print("For now, proceeding with current kernel...")

print("=" * 60 + "\n")

# Suppress warnings
import warnings
warnings.filterwarnings('ignore')


🚀 ==========================================================
JUPYTER KERNEL STARTUP DIAGNOSTICS
Kernel Python: /scratch2/f004ndc/RL-Decoder with SAE Features/phase6/venv/bin/python3
Version: 3.12.3 (main, Aug 14 2025, 17:47:21) [GCC 13.3.0]
Platform: linux

Running in Virtual Environment: ✅ YES



# 🧠 Interactive Reasoning Tracker - Jupyter Notebook

Visualize layer-by-layer SAE activations as your chosen model processes reasoning prompts.

---

## ⚡ QUICK START (Recommended)

Run this notebook with the automatic launcher script:

```bash
cd phase6
./run_jupyter.sh
```

This ensures the **virtual environment is properly activated** and all dependencies are installed.

---

## 📋 What You Can Do

- ✅ Select any of 4 frontier models (GPT2-medium, Phi-2, Gemma-2B, Pythia-1.4B)
- ✅ Enter any reasoning prompt
- ✅ Watch real-time heatmap of layer-by-layer activations  
- ✅ See top-8 features per layer with activation scores
- ✅ Understand what each layer is computing during reasoning

---

## ⏭️ Next Steps

1. **Cell 1** runs automatically - Check the kernel info
2. **Cell 2** - Verifies venv and imports core libraries
3. **Cell 3** - Loads ML libraries (transformers, SAE)
4. **Cell 4** - Model configurations (auto-loaded)
5. **Cell 5** - Initialize tracker
6. **Cells 6+** - Interactive controls (select model & run!)

---

In [2]:
# SETUP: Verify and ensure virtual environment is active

import sys
import os
from pathlib import Path
import subprocess

print("=" * 60)
print("🔍 VIRTUAL ENVIRONMENT SETUP")
print("=" * 60)
print()

# 1. Detect venv status
venv_active = hasattr(sys, 'real_prefix') or (hasattr(sys, 'base_prefix') and sys.base_prefix != sys.prefix)
python_executable = sys.executable
venv_path = Path(python_executable).parent.parent if venv_active else None

print(f"Virtual environment status: {'✅ ACTIVE' if venv_active else '⚠️  NOT DETECTED'}")
print(f"Python executable: {python_executable}")
if venv_active:
    print(f"Virtual env path: {venv_path}")
print()

# 2. If venv not detected, provide guidance
if not venv_active:
    print("⚠️  IMPORTANT: Virtual environment not detected!")
    print()
    print("TO FIX: Run this notebook with the launcher script:")
    print("  cd phase6")
    print("  ./run_jupyter.sh")
    print()
    print("If running manually, activate venv first:")
    print("  source venv/bin/activate")
    print("  jupyter notebook interactive_reasoning_tracker.ipynb")
    print()
    print("Attempting to find and use venv from current directory...")
    
    # Try to find venv in current directory
    current_dir = Path.cwd()
    venv_candidates = [current_dir / "venv", Path.home() / "phase6" / "venv"]
    
    venv_found = None
    for candidate in venv_candidates:
        if candidate.exists():
            venv_found = candidate
            print(f"  Found venv at: {venv_found}")
            break
    
    if not venv_found:
        print("  ⚠️  No venv found - will install packages globally to notebook kernel")
    print()

# 3. Import core libraries
print("📦 Importing core libraries...")
try:
    import torch
    import numpy as np
    import matplotlib.pyplot as plt
    import matplotlib.patches as mpatches
    from matplotlib.colors import Normalize, LinearSegmentedColormap
    from matplotlib.animation import FuncAnimation
    from IPython.display import HTML, display, clear_output, update_display
    import ipywidgets as widgets
    from ipywidgets import Output, VBox, HBox, Label, HTML as HTMLWidget
    import json
    import time
    from collections import defaultdict
    from typing import Dict, List, Tuple, Generator
    print("✅ All core libraries loaded successfully")
except ImportError as e:
    print(f"❌ Missing library: {e}")
    print()
    print("Installing dependencies using kernel's Python...")
    result = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "torch", "numpy", "matplotlib", "ipywidgets"],
        capture_output=True,
        text=True
    )
    if result.returncode == 0:
        print("✅ Dependencies installed successfully")
        print("Please restart kernel: Kernel → Restart in menu")
    else:
        print("❌ Installation failed:")
        print(result.stderr)

# 4. Add src to path
sys.path.insert(0, str(Path.cwd().parent / 'src'))

print()
print("=" * 60)
print("✅ SETUP COMPLETE - Ready to continue!")
print("=" * 60)
print()

🔍 VIRTUAL ENVIRONMENT SETUP

Virtual environment status: ✅ ACTIVE
Python executable: /scratch2/f004ndc/RL-Decoder with SAE Features/phase6/venv/bin/python3
Virtual env path: /scratch2/f004ndc/RL-Decoder with SAE Features/phase6/venv

📦 Importing core libraries...
✅ All core libraries loaded successfully

✅ SETUP COMPLETE - Ready to continue!



In [3]:
# Load transformers and SAE architecture with proper error handling

print("=" * 60)
print("📥 LOADING ML LIBRARIES")
print("=" * 60)
print()

import subprocess
import sys

# Try to import transformers
try:
    from transformers import AutoModelForCausalLM, AutoTokenizer
    print("✅ Transformers imported successfully")
except ImportError:
    print("⚠️  Transformers not found, installing...")
    result = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "transformers"],
        capture_output=True,
        text=True
    )
    if result.returncode == 0:
        from transformers import AutoModelForCausalLM, AutoTokenizer
        print("✅ Transformers installed and imported")
    else:
        print(f"❌ Failed to install: {result.stderr}")

# Try to import SAE architecture
try:
    from sae_architecture import SparseAutoencoder
    print("✅ SAE architecture imported successfully")
    have_sae_arch = True
except ImportError:
    print("⚠️  SAE architecture module not found")
    print("    Will use dummy SAEs for demonstration")
    have_sae_arch = False

print()
print(f"Using Python: {sys.executable}")
print(f"Python version: {sys.version.split()[0]}")
print()
print("=" * 60)
print("✅ ML LIBRARIES READY")
print("=" * 60)
print()

# Store flag for later use
import builtins
if not hasattr(builtins, 'tracker_config'):
    builtins.tracker_config = {}
builtins.tracker_config['have_sae_arch'] = have_sae_arch

📥 LOADING ML LIBRARIES

✅ Transformers imported successfully
✅ SAE architecture imported successfully

Using Python: /scratch2/f004ndc/RL-Decoder with SAE Features/phase6/venv/bin/python3
Python version: 3.12.3

✅ ML LIBRARIES READY



In [4]:
# Define model configurations
MODELS_CONFIG = {
    "gpt2-medium": {
        "model_id": "gpt2-medium",
        "num_layers": 24,
        "hidden_dim": 1024,
        "num_features": 8192  # 8x expansion
    },
    "phi-2": {
        "model_id": "microsoft/phi-2",
        "num_layers": 32,
        "hidden_dim": 2560,
        "num_features": 20480  # 8x expansion
    },
    "gemma-2b": {
        "model_id": "google/gemma-2b",
        "num_layers": 18,
        "hidden_dim": 2048,
        "num_features": 16384  # 8x expansion
    },
    "pythia-1.4b": {
        "model_id": "EleutherAI/pythia-1.4b",
        "num_layers": 24,
        "hidden_dim": 2048,
        "num_features": 16384  # 8x expansion
    }
}

# Layer semantics (from layer_semantics.json)
LAYER_SEMANTICS = {
    "gpt2-medium": {
        "0-3": "🟦 Input Tokenization - Converting text to tokens",
        "4-7": "🟩 Basic Syntax Detection - Punctuation and grammar",
        "8-11": "🟨 Semantic Composition - Word meanings in context",
        "12-15": "🟧 Reasoning Primitives - Arithmetic and logic",
        "16-19": "🟪 Higher-Order Reasoning - Planning and inference",
        "20-23": "🔴 Output Formulation - Response generation"
    },
    "phi-2": {
        "0-5": "🟦 Input Processing - Tokenization and embedding",
        "6-12": "🟩 Problem Decomposition - Breaking down reasoning",
        "13-19": "🟨 Semantic Analysis - Understanding relationships",
        "20-25": "🟧 Computation & Synthesis - Combining information",
        "26-31": "🔴 Output Generation - Response formulation"
    },
    "gemma-2b": {
        "0-4": "🟦 Input & Tokenization",
        "5-9": "🟩 Semantic Understanding",
        "10-13": "🟧 Reasoning & Computation",
        "14-17": "🔴 Output Generation"
    },
    "pythia-1.4b": {
        "0-5": "🟦 Input Processing",
        "6-11": "🟩 Semantic Understanding",
        "12-17": "🟧 Reasoning & Computation",
        "18-23": "🔴 Output Generation"
    }
}

print("✓ Model configurations loaded")
print(f"Available models: {', '.join(MODELS_CONFIG.keys())}")


✓ Model configurations loaded
Available models: gpt2-medium, phi-2, gemma-2b, pythia-1.4b


In [5]:
# Import necessary types and utilities for ReasoningTrackerNotebook class
import torch
from typing import Dict, List, Tuple, Generator
import time
from collections import defaultdict

class ReasoningTrackerNotebook:
    """Jupyter notebook version of reasoning tracker"""
    
    def __init__(self):
        self.model = None
        self.tokenizer = None
        self.saes = {}
        self.current_model_name = None
        self.hooks = []
        self.activations = {}
        self.feature_vocab = {}
        self.device = "cpu"
        self.model_dtype = torch.float32  # Default dtype, detected when model loads
        
    def load_model(self, model_name: str) -> bool:
        """Load model and tokenizer with fallback options"""
        try:
            config = MODELS_CONFIG[model_name]
            print(f"Loading {model_name}...")
            
            # Detect available device
            if torch.cuda.is_available():
                self.device = "cuda"
                print(f"  Using GPU: {torch.cuda.get_device_name(0)}")
            else:
                self.device = "cpu"
                print(f"  Using CPU (no GPU detected)")
            
            # Try loading with device_map first (requires accelerate)
            try:
                self.model = AutoModelForCausalLM.from_pretrained(
                    config["model_id"],
                    trust_remote_code=True,
                    device_map="auto"
                )
                print(f"  ✓ Loaded with device_map='auto'")
            except Exception as device_map_error:
                # Fallback: try without device_map
                if "accelerate" in str(device_map_error).lower():
                    print(f"  ⚠ device_map requires accelerate, trying without...")
                    self.model = AutoModelForCausalLM.from_pretrained(
                        config["model_id"],
                        trust_remote_code=True
                    )
                    if torch.cuda.is_available():
                        self.model = self.model.to(self.device)
                    print(f"  ✓ Loaded without device_map")
                else:
                    raise
            
            # ✅ FIX: Detect model dtype (critical for float16 models like phi-2)
            for param in self.model.parameters():
                self.model_dtype = param.dtype
                break
            print(f"  Model dtype: {self.model_dtype}")
            
            # Load tokenizer
            self.tokenizer = AutoTokenizer.from_pretrained(
                config["model_id"],
                trust_remote_code=True
            )
            
            self.current_model_name = model_name
            print(f"✓ {model_name} loaded successfully ({config['num_layers']} layers)")
            return True
            
        except Exception as e:
            print(f"❌ Error loading {model_name}:")
            print(f"   {str(e)[:200]}")
            print(f"\n   To fix accelerate issues, run:")
            print(f"   pip install accelerate")
            return False
    
    def load_saes(self, model_name: str) -> bool:
        """Load SAE checkpoints for model"""
        try:
            # ✅ FIX: Try multiple possible paths for phase5_results
            # Handles different working directories when running jupyter
            possible_paths = [
                Path.cwd() / "phase5_results",  # If running from project root
                Path.cwd().parent / "phase5_results",  # If running from phase6
                Path("/scratch2/f004ndc/RL-Decoder with SAE Features") / "phase5_results",  # Absolute fallback
            ]
            
            sae_dir = None
            for path_candidate in possible_paths:
                # Prefer v2 (fixed: use_relu=True) over v1 (broken)
                for subdir in ["multilayer_transfer_v2", "multilayer_transfer"]:
                    candidate = path_candidate / subdir / "saes"
                    if candidate.exists():
                        sae_dir = candidate
                        print(f"  Found SAE directory: {sae_dir}")
                        break
                if sae_dir is not None:
                    break
            
            if sae_dir is None:
                print(f"  ⚠ SAE directory not found in:")
                for path_candidate in possible_paths:
                    print(f"    - {path_candidate / 'multilayer_transfer' / 'saes'}")
                print(f"  Falling back to dummy SAEs")
                self._create_dummy_saes(model_name)
                print(f"⚠ Using dummy SAEs for {model_name}")
                return True
            
            config = MODELS_CONFIG[model_name]
            num_layers = config["num_layers"]
            
            print(f"Loading {num_layers} SAEs for {model_name}...")
            loaded_count = 0
            
            for layer_id in range(num_layers):
                checkpoint_path = sae_dir / f"{model_name}_layer{layer_id}_sae.pt"
                if checkpoint_path.exists():
                    try:
                        # weights_only=False required: checkpoints contain PosixPath objects
                        sae_state = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
                        state_dict = sae_state["model_state_dict"]
                        # Read actual dims from checkpoint (may differ from MODELS_CONFIG)
                        actual_input_dim = state_dict["encoder.weight"].shape[1]
                        actual_num_features = state_dict["encoder.weight"].shape[0]
                        actual_config = dict(config)
                        actual_config["hidden_dim"] = actual_input_dim
                        actual_config["num_features"] = actual_num_features
                        sae = self._create_dummy_sae(actual_config)
                        sae.load_state_dict(state_dict)
                        sae = sae.to(device=self.device, dtype=self.model_dtype)
                        self.saes[layer_id] = sae
                        loaded_count += 1
                    except Exception as e:
                        print(f"  Warning: Could not load layer {layer_id} SAE: {e}")
                        self.saes[layer_id] = self._create_dummy_sae(config)
                else:
                    # File doesn't exist, use dummy SAE
                    self.saes[layer_id] = self._create_dummy_sae(config)
            
            if loaded_count > 0:
                print(f"✓ Loaded {loaded_count} real SAEs + {num_layers - loaded_count} dummy SAEs for {model_name}")
            else:
                print(f"⚠ Using dummy SAEs for {model_name} (real checkpoint files not found)")
            
            return True
        except Exception as e:
            print(f"❌ Error loading SAEs: {e}")
            self._create_dummy_saes(model_name)
            return False
    
    def _create_dummy_sae(self, config):
        """Create dummy SAE for testing with matching dtype and device"""
        class DummySAE(torch.nn.Module):
            def __init__(self, input_dim, num_features):
                super().__init__()
                # Create Linear layers in default dtype first
                self.encoder = torch.nn.Linear(input_dim, num_features)
                self.decoder = torch.nn.Linear(num_features, input_dim)
            
            def forward(self, x):
                latents = self.encoder(x)
                return self.decoder(latents)
        
        # Create SAE in default dtype
        sae = DummySAE(config["hidden_dim"], config["num_features"])
        
        # ✅ FIX: Move to correct device and dtype (handles float32, float16, bfloat16, etc.)
        # This prevents both "different devices" AND "dtype mismatch" errors
        # Uses .to() which is more robust than dtype parameter in constructor
        sae = sae.to(device=self.device, dtype=self.model_dtype)
        return sae
    
    def _create_dummy_saes(self, model_name: str):
        """Create all dummy SAEs for a model"""
        config = MODELS_CONFIG[model_name]
        for layer_id in range(config["num_layers"]):
            self.saes[layer_id] = self._create_dummy_sae(config)
    
    def _setup_hooks(self):
        """Setup forward hooks to capture activations"""
        # Remove old hooks
        for hook in self.hooks:
            hook.remove()
        self.hooks.clear()
        
        config = MODELS_CONFIG[self.current_model_name]
        
        def make_hook(layer_id):
            def hook(module, input, output):
                if isinstance(output, tuple):
                    self.activations[layer_id] = output[0].detach()
                else:
                    self.activations[layer_id] = output.detach()
            return hook
        
        # Register hooks on full transformer blocks (matching the training capture script,
        # which hooked transformer.h[i] / model.layers[i], not just the MLP sub-module)
        try:
            for layer_id in range(config["num_layers"]):
                try:
                    module = self.model.transformer.h[layer_id]
                except:
                    try:
                        module = self.model.model.layers[layer_id]
                    except:
                        try:
                            module = self.model.gpt_neox.layers[layer_id]
                        except:
                            continue
                
                hook = module.register_forward_hook(make_hook(layer_id))
                self.hooks.append(hook)
        except Exception as e:
            print(f"Warning: Could not register all hooks: {e}")
    
    def run_reasoning(self, prompt: str, max_tokens: int = 30) -> Generator:
        """Generate tokens and capture layer activations"""
        if self.model is None:
            print("❌ Model not loaded")
            return
        
        self._setup_hooks()
        
        # Tokenize prompt
        input_ids = self.tokenizer.encode(prompt, return_tensors="pt")
        # ✅ FIX: Move input_ids to same device as model
        input_ids = input_ids.to(self.device)
        
        config = MODELS_CONFIG[self.current_model_name]
        generated_tokens = []
        
        with torch.no_grad():
            for token_pos in range(max_tokens):
                self.activations.clear()
                
                # Generate next token
                outputs = self.model(input_ids)
                next_token_logits = outputs.logits[0, -1, :]
                next_token_id = torch.argmax(next_token_logits).unsqueeze(0).unsqueeze(0)
                input_ids = torch.cat([input_ids, next_token_id], dim=-1)
                
                token_str = self.tokenizer.decode([next_token_id.item()])
                generated_tokens.append(token_str)
                
                # Collect layer data
                layer_data = []
                for layer_id in range(config["num_layers"]):
                    if layer_id in self.activations and layer_id in self.saes:
                        activation = self.activations[layer_id]
                        sae = self.saes[layer_id]
                        
                        # ✅ FIX: Ensure activation is on same device and dtype as SAE
                        activation = activation.to(device=self.device, dtype=self.model_dtype)
                        
                        # Compute SAE output
                        with torch.no_grad():
                            latents = sae.encoder(activation)
                            decoded = sae.decoder(latents)
                            recon_loss = ((activation - decoded) ** 2).mean().item()
                        
                        # Get top features
                        feature_scores = latents.abs().squeeze()
                        if len(feature_scores.shape) > 1:
                            feature_scores = feature_scores.max(dim=0)[0]
                        
                        top_k = min(8, len(feature_scores))
                        top_indices = torch.topk(feature_scores, top_k)[1].cpu().numpy()
                        top_scores = torch.topk(feature_scores, top_k)[0].cpu().numpy()
                        
                        top_features = [
                            {"id": int(idx), "score": float(score)}
                            for idx, score in zip(top_indices, top_scores)
                        ]
                        
                        layer_data.append({
                            "layer": layer_id,
                            "token_str": token_str,
                            "sae_loss": recon_loss,
                            "top_features": top_features,
                            "layer_meaning": self._get_layer_meaning(layer_id)
                        })
                
                yield {
                    "token_pos": token_pos,
                    "layers": layer_data,
                    "full_text": prompt + "".join(generated_tokens)
                }
                
                time.sleep(0.01)  # Small delay for animation
    
    def _get_layer_meaning(self, layer_id: int) -> str:
        """Get semantic meaning of layer"""
        if self.current_model_name not in LAYER_SEMANTICS:
            return f"Layer {layer_id}"
        
        meanings = LAYER_SEMANTICS[self.current_model_name]
        for range_str, meaning in meanings.items():
            start, end = map(int, range_str.split("-"))
            if start <= layer_id <= end:
                return meaning
        return f"Layer {layer_id}"

# Create tracker instance
tracker = ReasoningTrackerNotebook()
print("✓ Tracker initialized")


✓ Tracker initialized


---

## Interactive Controls

Use the controls below to explore reasoning activations:

In [6]:
# Create interactive widgets
model_dropdown = widgets.Dropdown(
    options=list(MODELS_CONFIG.keys()),
    value="gpt2-medium",
    description="Model:",
    layout=widgets.Layout(width="200px")
)

prompt_input = widgets.Text(
    value="What is 2+2?",
    placeholder="Enter your prompt here...",
    description="Prompt:",
    layout=widgets.Layout(width="500px")
)

max_tokens_input = widgets.IntSlider(
    value=20,
    min=1,
    max=100,
    step=1,
    description="Max Tokens:",
    layout=widgets.Layout(width="300px")
)

run_button = widgets.Button(
    description="▶ Run Reasoning",
    button_style="success",
    tooltip="Start the reasoning process",
    layout=widgets.Layout(width="150px", height="40px")
)

status_output = widgets.Output()
visualization_output = widgets.Output()
stream_output = widgets.Output()

print("✓ Widgets created")

✓ Widgets created


In [7]:
# Define callback functions for interactive controls
from collections import defaultdict

def on_model_change(change):
    """Handle model selection change"""
    model_name = change["new"]
    with status_output:
        clear_output()
        print(f"Loading {model_name}...")
        if tracker.load_model(model_name):
            tracker.load_saes(model_name)
            config = MODELS_CONFIG[model_name]
            print(f"✓ Ready! ({config['num_layers']} layers, {config['num_features']} features)")
        else:
            print(f"❌ Failed to load {model_name}")

def on_run_button_click(b):
    """Handle run button click"""
    if tracker.model is None:
        with status_output:
            clear_output()
            print("Please load a model first by selecting from the dropdown.")
        return
    
    prompt = prompt_input.value
    max_tokens = max_tokens_input.value
    
    with status_output:
        clear_output()
        print(f"Running reasoning for: '{prompt}'")
        print(f"Model: {tracker.current_model_name} | Max tokens: {max_tokens}")
    
    with stream_output:
        clear_output()
        print("🔄 Running inference...")
    
    # Prepare visualization
    all_layer_data = defaultdict(list)
    generated_text = prompt
    
    try:
        for result in tracker.run_reasoning(prompt, max_tokens):
            # Collect data
            for layer_info in result["layers"]:
                layer_id = layer_info["layer"]
                all_layer_data[layer_id].append(layer_info["sae_loss"])
            
            generated_text = result["full_text"]
            
            # Update stream
            with stream_output:
                clear_output(wait=True)
                print("Generated text so far:")
                print(f"\n{generated_text}\n")
                print(f"Progress: {result['token_pos']+1}/{max_tokens} tokens")
        
        # Create final visualization
        with visualization_output:
            clear_output()
            _create_visualization(all_layer_data)
        
        with status_output:
            clear_output()
            print(f"✓ Complete! Generated: {generated_text[len(prompt):]}")
    except Exception as e:
        with status_output:
            clear_output()
            print(f"❌ Error: {e}")

# Register callbacks
model_dropdown.observe(on_model_change, names="value")
run_button.on_click(on_run_button_click)

print("✓ Callbacks registered")

✓ Callbacks registered


In [8]:
# Auto-load default model on startup
print("\nAuto-loading default model...")
with status_output:
    clear_output()
    print(f"Loading gpt2-medium...")
    if tracker.load_model("gpt2-medium"):
        tracker.load_saes("gpt2-medium")
        config = MODELS_CONFIG["gpt2-medium"]
        print(f"✓ Ready! ({config['num_layers']} layers, {config['num_features']} features)")
    else:
        print(f"❌ Failed to load gpt2-medium")


Auto-loading default model...


In [9]:
# Visualization function for layer activations
import numpy as np
import matplotlib.pyplot as plt

def _create_visualization(layer_data):
    """Create visualization of layer activations"""
    if not layer_data:
        print("No data to visualize")
        return
    
    # Convert to visualization format
    num_layers = len(layer_data)
    layer_ids = sorted(layer_data.keys())
    max_tokens = max(len(v) for v in layer_data.values())
    
    # Create 2D array: layers × tokens (raw reconstruction losses)
    heatmap_data = np.zeros((num_layers, max_tokens))
    for i, layer_id in enumerate(layer_ids):
        losses = layer_data[layer_id]
        heatmap_data[i, :len(losses)] = losses
    
    # Create figure
    fig, ax = plt.subplots(figsize=(12, 6))
    
    # Reverse SAE loss to quality (high loss = low quality = dark)
    quality_data = 1.0 / (1.0 + heatmap_data)
    
    # Color map: blue (good) → orange (bad)
    # ✅ FIX: Use matplotlib 3.7+ compatible colormap access (fixes deprecation warning)
    try:
        cmap = plt.colormaps.get_cmap('RdYlGn_r')  # Matplotlib 3.7+
    except AttributeError:
        cmap = plt.cm.get_cmap('RdYlGn_r')  # Fallback for older versions
    im = ax.imshow(quality_data, aspect='auto', cmap=cmap, vmin=0.0, vmax=1.0, interpolation='nearest')
    
    # Labels
    ax.set_ylabel("Layer", fontsize=12)
    ax.set_xlabel("Token Position", fontsize=12)
    ax.set_title(f"SAE Reconstruction Quality Heatmap", fontsize=14, fontweight='bold')
    
    # Set layer labels
    layer_labels = [f"L{lid}" for lid in layer_ids]
    ax.set_yticks(range(num_layers))
    ax.set_yticklabels(layer_labels, fontsize=9)
    
    # Colorbar
    cbar = plt.colorbar(im, ax=ax, label="Reconstruction Quality")
    
    plt.tight_layout()
    plt.show()

print("✓ Visualization function ready")

✓ Visualization function ready


In [10]:
# Display the interactive interface
from ipywidgets import VBox, HBox, Label, HTML as HTMLWidget
from IPython.display import display

print("="*60)
print("🧠 INTERACTIVE REASONING TRACKER")
print("="*60)
print()
print("Step 1: Select a model from the dropdown")
print("Step 2: Enter your reasoning prompt")
print("Step 3: Adjust max tokens if desired")
print("Step 4: Click 'Run Reasoning' to start")
print()
print("="*60)
print()

# Create layout
controls = VBox([
    HBox([Label("Model:"), model_dropdown]),
    HBox([Label("Prompt:"), prompt_input]),
    HBox([Label("Max Tokens:"), max_tokens_input]),
    HBox([run_button]),
])

# Display everything
display(controls)
display(HTMLWidget("<hr>"))
display(status_output)
display(HTMLWidget("<h3>Visualization</h3>"))
display(visualization_output)
display(HTMLWidget("<h3>Generation Stream</h3>"))
display(stream_output)

print("\n✓ Interactive interface ready! Start with Step 1 above.")

🧠 INTERACTIVE REASONING TRACKER

Step 1: Select a model from the dropdown
Step 2: Enter your reasoning prompt
Step 3: Adjust max tokens if desired
Step 4: Click 'Run Reasoning' to start




HTML(value='<hr>')

Output()

HTML(value='<h3>Visualization</h3>')

Output()

HTML(value='<h3>Generation Stream</h3>')

Output()


✓ Interactive interface ready! Start with Step 1 above.


---

## Usage Examples

Try these prompts to explore different reasoning patterns:

### 1. Simple Arithmetic
```
What is 17 + 25?
```

**What to look for:**
- Early layers: Tokenization of numbers
- Middle layers: Arithmetic computation
- Late layers: Output generation

### 2. Logic Problem
```
All cats are animals. Fluffy is a cat. Is Fluffy an animal?
```

**What to look for:**
- Distributed activation across layers
- Feature activation for entities vs relationships

### 3. Multi-step Reasoning
```
I have 3 apples. I eat 1. How many remain?
```

**What to look for:**
- Sequential layer activation
- State tracking across layers

---

## How to Interpret Results

### Heatmap Colors
- 🟩 **Green (high quality)**: SAE well reconstructs layer activation
- 🟨 **Yellow (medium quality)**: Some information loss
- 🟥 **Red (low quality)**: Poor reconstruction (likely complex features)

### Patterns to Watch
- **Vertical stripes**: Same feature active across all tokens
- **Horizontal bands**: Feature concentrated in specific layer
- **Diagonal**: Features activate sequentially through layers
- **Random noise**: Features sparse/unique per token

---

## Next Steps

Once you've explored:
1. Compare patterns between different models
2. Try longer sequences (increase max_tokens)
3. Look for universal features across models
4. See Phase 6.1 for feature ablation tools